In [ ]:
import _referAsMain

import math
import time
import random
import sys
import numpy
from IPython.display import SVG, display
from holo.pointers import Pointer
from holo.profilers import Profiler
from holo.prettyFormats import prettyPrint, prettyTime
print(sys.version_info)

import torch
from torch.utils.data import DataLoader

from paths_cfg import TOKENIZER_SAVE_DIRECTORY, OUR_DATASET_DIRECTORY
from dataset import svg_dataset
import tokenizer_pfe.tokenizer_project as tokenizerLib
import metrics.metrics

from LLM.model import Model, GenerationStats
import LLM.nanochat.gpt as nanoChatModel
from LLM.nanochat.common import compute_init, autodetect_device_type

In [ ]:
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

In [ ]:
import importlib
import LLM.model
_ = importlib.reload(LLM.model)
from LLM.model import Model

In [ ]:
try:
    torch.cuda.empty_cache()
    del model # type: ignore
    torch.cuda.empty_cache()
except Exception: pass
model = Model.load("tmp-tests", versionID=5, device=torch.device("cuda:0"))
model.show_infos()

In [ ]:
try:
    if "dataset" in globals() : raise FileExistsError
    dataset = svg_dataset.SVGDataset(
        OUR_DATASET_DIRECTORY, context_size=model.context_size,
        tokenizer=model.tokenizer.encode, decoder=model.tokenizer.decode)
except FileExistsError: pass

In [ ]:
statsPtr: Pointer[GenerationStats] = Pointer()
results = []
for txt in model.generate_flow(
        start=None, decode_batch=256, temperature=1, top_k=5, 
        max_tokens=32000, max_time=2*60, statsPtr=statsPtr):
    results.append(txt)
    print(txt, end="", flush=True)

In [ ]:
print(statsPtr.value)
print(f" -> {statsPtr.value.nb_tokens / statsPtr.value.gen_time:.2f} tokens/sec")

In [ ]:
print(results)

In [ ]:
print("".join(results))

In [ ]:
display(SVG(data="".join(results)))